# 01 — Clustering de la consommation énergétique

Objectif : regrouper les 500 clients en clusters homogènes à partir de **features construites sur mesure**.

Le fichier `RES2-6-9-labels.csv` est la référence du professeur — nous l'utilisons **uniquement à la fin** pour valider nos résultats.

### Pipeline
1. Chargement et préparation des données
2. Agrégation journalière + détection des jours actifs
3. Features d'activité globale
4. Features de régularité (rachas et gaps)
5. Features hebdomadaires (semaine vs week-end)
6. Features saisonnières (ratios normalisés)
7. Assemblage et normalisation
8. K-Means avec sélection de K
9. Visualisation PCA
10. Profils intrajournaliers par cluster
11. Comparaison avec la référence du professeur

In [ ]:

# Imports

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score, adjusted_rand_score
from sklearn.decomposition import PCA
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (14, 6)
sns.set_style('whitegrid')
RANDOM_STATE = 42

## 1. Chargement des données

In [ ]:

# Chargement du CSV et création des colonnes temporelles de base


BASE_DIR = Path(__file__).resolve().parent.parent

DATA_PATH = BASE_DIR / "Data" / "RES2-6-9.csv"
LABELS_PATH = BASE_DIR / "Data" / "RES2-6-9-labels.csv"   # référence prof — utilisée seulement à la fin

COL_PDL = 'pdl_id'
COL_DT  = 'horodate'
COL_PWR = 'puissance_w'

raw = pd.read_csv(DATA_PATH)
raw.columns = [COL_PDL, COL_DT, COL_PWR]

# Conversion datetime avec timezone Europe/Paris
raw[COL_DT] = pd.to_datetime(raw[COL_DT], utc=True, errors='coerce')
if raw[COL_DT].dt.tz is None:
    raw[COL_DT] = raw[COL_DT].dt.tz_localize('Europe/Paris')
else:
    raw[COL_DT] = raw[COL_DT].dt.tz_convert('Europe/Paris')

# Nettoyage des valeurs manquantes
df = raw.dropna(subset=[COL_PDL, COL_DT, COL_PWR]).copy()
df[COL_PWR] = pd.to_numeric(df[COL_PWR], errors='coerce')
df = df.dropna(subset=[COL_PWR])

# Colonnes temporelles
df['date']       = df[COL_DT].dt.date
df['dow']        = df[COL_DT].dt.dayofweek          # 0=lundi, 6=dimanche
df['is_weekend'] = df['dow'] >= 5
df['hh_index']   = df[COL_DT].dt.hour * 2 + df[COL_DT].dt.minute // 30  # 0..47

print(f'Clients       : {df[COL_PDL].nunique()}')
print(f'Période       : {df[COL_DT].min().date()} -> {df[COL_DT].max().date()}')
print(f'Mesures total : {len(df):,}')

## 2. Agrégation journalière et détection des jours actifs

In [ ]:

# Agrégation par (client, jour)

# Chaque mesure = 30 min → énergie = puissance (kW) × 0.5 h = kWh
daily = (
    df.assign(energy_kwh_step=df[COL_PWR] * 0.5)
    .groupby([COL_PDL, 'date'], as_index=False)
    .agg(
        daily_kwh    =('energy_kwh_step', 'sum'),   # énergie totale (kWh)
        daily_mean_kw=(COL_PWR,           'mean'),  # puissance moyenne
        daily_max_kw =(COL_PWR,           'max'),   # puissance de pointe
        n_steps      =(COL_PWR,           'size'),  # nombre de mesures (≈48)
    )
)

print(f'Lignes daily : {len(daily):,}  (clients x jours)')
daily.head()

In [ ]:

# Seuil d'activité personnalisé par client (percentile 20 des jours > 0)

# Évite d'appliquer un seuil global inadapté aux petits consommateurs.

def q20_positive(s):
    # Percentile 20 des valeurs strictement positives
    s = s[s > 0]
    return s.quantile(0.2) if len(s) > 0 else np.nan

daily['th_pdl'] = (
    daily.groupby(COL_PDL)['daily_kwh']
    .transform(q20_positive)
)

# fillna(False) : client sans aucun jour positif -> jamais actif
daily['is_active_day'] = (daily['daily_kwh'] >= daily['th_pdl']).fillna(False)

print(f'Taux d activite moyen : {daily["is_active_day"].mean():.1%}')
daily[['pdl_id','date','daily_kwh','th_pdl','is_active_day']].head(8)

## 3. Feature 1 — Activité globale

In [ ]:

# Résumé statistique de l'activité de chaque client

# active_day_rate : fréquence d'utilisation (0-1)
# mean_daily_kwh  : consommation journalière moyenne
# p95_daily_kwh   : consommation lors des journées intenses
# cv_daily_kwh    : irrégularité (coefficient de variation = std/mean)

activity = (
    daily.groupby(COL_PDL, as_index=False)
    .agg(
        n_days         =(COL_PDL,         'size'),
        n_active_days  =('is_active_day', 'sum'),
        active_day_rate=('is_active_day', 'mean'),
        mean_daily_kwh =('daily_kwh',     'mean'),
        p95_daily_kwh  =('daily_kwh',     lambda s: s.quantile(0.95)),
        cv_daily_kwh   =('daily_kwh',     lambda s: s.std() / s.mean() if s.mean() != 0 else np.nan),
    )
)

print(f'activity shape : {activity.shape}')
activity.head()

## 4. Feature 2 — Rachas et gaps d'activité

In [ ]:

# Séquences de jours actifs (runs) et inactifs (gaps)

# n_runs       : nombre total de périodes actives
# mean_run_len : durée moyenne des périodes actives consécutives
# max_run_len  : plus longue période active
# mean_gap_len : durée moyenne des pauses entre périodes actives
# max_gap_len  : plus longue pause (absence prolongée)

def runs_and_gaps(active_series):
    # Retourne une pd.Series (compatible avec .apply().unstack())
    runs, gaps = [], []
    run, gap   = 0, 0
    for v in active_series.astype(bool):
        if v:
            run += 1
            if gap > 0:
                gaps.append(gap)
                gap = 0
        else:
            gap += 1
            if run > 0:
                runs.append(run)
                run = 0
    if run > 0: runs.append(run)
    if gap > 0: gaps.append(gap)
    return pd.Series({
        'n_runs'      : len(runs),
        'mean_run_len': float(np.mean(runs)) if runs else 0.0,
        'max_run_len' : float(np.max(runs))  if runs else 0.0,
        'mean_gap_len': float(np.mean(gaps)) if gaps else 0.0,
        'max_gap_len' : float(np.max(gaps))  if gaps else 0.0,
    })

# Tri chronologique obligatoire avant de calculer les rachas
# .unstack() convertit la Series de Series en DataFrame (évite les problèmes de MultiIndex)
runs_stats = (
    daily.sort_values([COL_PDL, 'date'])
    .groupby(COL_PDL)['is_active_day']
    .apply(runs_and_gaps)
    .unstack()
    .reset_index()
)

print(f'runs_stats shape : {runs_stats.shape}')
runs_stats.head()

## 5. Feature 3 — Pattern hebdomadaire

In [ ]:

# Comportement différencié semaine / week-end

# RS (Résidence Secondaire) : forte consommation le week-end (présents uniquement en vacances/week-end)
# RP (Résidence Principale) : consommation plus régulière semaine/week-end (occupants à domicile toute la semaine)
# -> feature très discriminante entre les deux types

daily_dt          = pd.to_datetime(daily['date'])
daily['dow']      = daily_dt.dt.dayofweek
daily['is_weekend']= daily['dow'] >= 5

week_pattern = (
    daily
    .groupby([COL_PDL, 'is_weekend'], as_index=False)
    .agg(
        active_rate=('is_active_day', 'mean'),
        mean_kwh   =('daily_kwh',     'mean'),
    )
    .pivot(index=COL_PDL, columns='is_weekend')
)

# Renommage des colonnes après pivot
week_pattern.columns = [
    f"{stat}_{'weekend' if is_we else 'weekday'}"
    for stat, is_we in week_pattern.columns
]
week_pattern = week_pattern.reset_index()

print(f'week_pattern shape : {week_pattern.shape}')
week_pattern.head()

## 6. Feature 4 — Pattern saisonnier (ratios normalisés)

In [ ]:

# Ratios saisonniers normalisés par la consommation moyenne globale

# On calcule r_winter = mean_winter / mean_global (idem summer, mid)
# Cela permet de comparer la FORME saisonnière indépendamment du volume :
#   r_winter > 1 -> consomme plus l'hiver que sa propre moyenne
#   r_summer < 1 -> consomme moins l'été que sa propre moyenne

def season_from_month(m):
    if m in (12, 1, 2): return 'winter'
    if m in (6, 7, 8):  return 'summer'
    return 'mid'

daily2          = daily.copy()
daily2['date_ts']= pd.to_datetime(daily2['date'])
daily2['month'] = daily2['date_ts'].dt.month
daily2['season']= daily2['month'].map(season_from_month)

season_stats = (
    daily2.groupby([COL_PDL, 'season'], as_index=False)
    .agg(mean_daily_kwh=('daily_kwh', 'mean'))
    .pivot(index=COL_PDL, columns='season', values='mean_daily_kwh')
    .reset_index()
)

# Colonnes manquantes -> 0 (client sans données dans une saison)
for c in ['winter', 'summer', 'mid']:
    if c not in season_stats.columns:
        season_stats[c] = 0.0

# Moyenne globale par client (dénominateur des ratios)
global_mean = (
    daily2.groupby(COL_PDL, as_index=False)
    .agg(mean_daily_kwh_global=('daily_kwh', 'mean'))
)
season_stats = season_stats.merge(global_mean, on=COL_PDL, how='left', validate='one_to_one')

# Calcul des ratios (eps pour éviter la division par zéro)
eps = 1e-9
season_stats['r_global'] = 1.0   # référence (par définition)
season_stats['r_mid']    = season_stats['mid']    / (season_stats['mean_daily_kwh_global'] + eps)
season_stats['r_summer'] = season_stats['summer'] / (season_stats['mean_daily_kwh_global'] + eps)
season_stats['r_winter'] = season_stats['winter'] / (season_stats['mean_daily_kwh_global'] + eps)

# On garde uniquement les ratios (pas les valeurs absolues)
season_stats = season_stats[[COL_PDL, 'r_global', 'r_mid', 'r_summer', 'r_winter']]

print(f'season_stats shape : {season_stats.shape}')
season_stats.head()

## 7. Assemblage et features dérivées

In [ ]:

# Fusion de toutes les features : 1 ligne par client

features_pdl = (
    activity
    .merge(runs_stats,   on=COL_PDL, how='left', validate='one_to_one')
    .merge(week_pattern, on=COL_PDL, how='left', validate='one_to_one')
    .merge(season_stats, on=COL_PDL, how='left', validate='one_to_one')
)

assert features_pdl[COL_PDL].is_unique, 'ERREUR : plusieurs lignes par client'
print(f'OK : 1 ligne par client — {len(features_pdl)} clients')


# Features dérivées : amplitude et direction de la saisonnalité

# seasonality_amp    : écart max-min entre saisons (forte valeur = client très saisonnier)
# winter_minus_summer: positif -> hiver > été (chauffage électrique probable)
#                      négatif -> été > hiver (climatisation ou comportement atypique)
features_pdl['seasonality_amp']    = (
    features_pdl[['r_mid','r_summer','r_winter']].max(axis=1) -
    features_pdl[['r_mid','r_summer','r_winter']].min(axis=1)
)
features_pdl['winter_minus_summer'] = features_pdl['r_winter'] - features_pdl['r_summer']

print(f'Matrice features : {features_pdl.shape}')
features_pdl.head()

In [ ]:

# Sélection explicite des features pour le clustering

# Liste identique à celle du professeur : 19 features
# On exclut n_days, n_active_days (redondants avec active_day_rate)
FEATURE_COLS = [
    'active_day_rate', 'n_runs', 'mean_run_len', 'max_run_len',
    'mean_gap_len', 'max_gap_len',
    'mean_daily_kwh', 'p95_daily_kwh', 'cv_daily_kwh',
    'active_rate_weekday', 'active_rate_weekend',
    'mean_kwh_weekday', 'mean_kwh_weekend',
    'winter_minus_summer', 'seasonality_amp',
    'r_global', 'r_mid', 'r_summer', 'r_winter',
]

X = features_pdl[FEATURE_COLS].copy()
# Remplacement des valeurs infinies et NaN par 0
X = X.replace([np.inf, -np.inf], np.nan).fillna(0.0)

# StandardScaler : centre et réduit chaque feature (moyenne=0, std=1)
# Indispensable pour K-Means qui est sensible aux différences d'échelle
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

print(f'Matrice normalisée : {X_scaled.shape}  ({len(FEATURE_COLS)} features)')

## 8. Choix du nombre de clusters K

In [ ]:

# Score de silhouette pour K de 2 à 6 puis K=10 imposé

scores = {}
for k in range(2, 11):
    km     = KMeans(n_clusters=k, n_init=20, random_state=RANDOM_STATE)
    labels = km.fit_predict(X_scaled)
    scores[k] = silhouette_score(X_scaled, labels)

best_k = max(scores, key=scores.get)
print(f'Silhouette par K : {scores}')
print(f'Meilleur K selon silhouette : {best_k}')

fig, ax = plt.subplots(figsize=(10, 4))
ax.plot(list(scores.keys()), list(scores.values()), 'rs-', linewidth=2, markersize=8)
ax.axvline(x=10, color='blue', linestyle='--', label='K=10 (choix final)')
ax.axvline(x=best_k, color='green', linestyle='--', label=f'K={best_k} (silhouette max)')
ax.set_xlabel('K', fontsize=12)
ax.set_ylabel('Score de silhouette', fontsize=12)
ax.set_title('Silhouette score par nombre de clusters', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.savefig('fig_01_silhouette.png', dpi=150, bbox_inches='tight')
plt.show()

## 9. Clustering final K-Means (K=10)

In [ ]:

# Clustering final avec K=10 et n_init=50

# n_init=50 : 50 initialisations aléatoires -> solution plus stable et reproductible
# K=10 est fixé pour correspondre à la référence du professeur
K_FINAL = 10
kmeans  = KMeans(n_clusters=K_FINAL, n_init=50, random_state=RANDOM_STATE)
features_pdl['cluster'] = kmeans.fit_predict(X_scaled)

sil_final = silhouette_score(X_scaled, features_pdl['cluster'])
print(f'Score de silhouette final (K={K_FINAL}) : {sil_final:.3f}')
print()
print('Distribution des clusters :')
print(features_pdl['cluster'].value_counts().sort_index().to_string())

features_pdl[['pdl_id','cluster']].head()

## 10. Visualisation PCA

In [ ]:

# Projection PCA 2D : visualise la séparation des clusters

pca    = PCA(n_components=2, random_state=RANDOM_STATE)
Z      = pca.fit_transform(X_scaled)
var_exp= pca.explained_variance_ratio_

pca_df = pd.DataFrame({
    'PC1'    : Z[:,0],
    'PC2'    : Z[:,1],
    'cluster': features_pdl['cluster'].astype(str),
})

palette = plt.cm.tab10(np.linspace(0, 1, K_FINAL))
fig, ax = plt.subplots(figsize=(10, 7))
for c in range(K_FINAL):
    mask = features_pdl['cluster'] == c
    ax.scatter(Z[mask,0], Z[mask,1], label=f'Cluster {c}',
               color=palette[c], alpha=0.8, s=50)

ax.set_xlabel(f'PC1 ({var_exp[0]:.1%} variance)')
ax.set_ylabel(f'PC2 ({var_exp[1]:.1%} variance)')
ax.set_title('Clustering K-Means — projection PCA 2D', fontsize=14, fontweight='bold')
ax.legend(loc='best', fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig('fig_02_pca.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Variance totale expliquée : {sum(var_exp):.1%}')

## 11. Profils intrajournaliers par cluster

In [ ]:

# Profil moyen demi-horaire par cluster (courbe de charge)

# On rattache le cluster à chaque mesure brute pour tracer
# la courbe de charge caractéristique de chaque groupe.

df_cluster = df.merge(
    features_pdl[[COL_PDL, 'cluster']], on=COL_PDL, how='left', validate='many_to_one'
)

mean_intraday = (
    df_cluster.groupby(['cluster', 'hh_index'], as_index=False)
    .agg(mean_kw=(COL_PWR, 'mean'))
)

x_ticks  = list(range(0, 48, 4))
x_labels = [f'{h:02d}:00' for h in range(0, 24, 2)]

fig, axes = plt.subplots(2, 5, figsize=(22, 9))
axes = axes.flatten()

for c in range(K_FINAL):
    sub = mean_intraday[mean_intraday['cluster'] == c]
    n_c = (features_pdl['cluster'] == c).sum()
    axes[c].plot(sub['hh_index'], sub['mean_kw'], color=palette[c], linewidth=2.5)
    axes[c].fill_between(sub['hh_index'], 0, sub['mean_kw'], color=palette[c], alpha=0.2)
    axes[c].set_title(f'Cluster {c}  (n={n_c})', fontsize=11, fontweight='bold', color=palette[c])
    axes[c].set_xticks(x_ticks)
    axes[c].set_xticklabels(x_labels, fontsize=7, rotation=45)
    axes[c].set_ylabel('kW moyen', fontsize=8)

plt.suptitle('Profil intrajournalier moyen par cluster (kW)', fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('fig_03_intraday_profiles.png', dpi=150, bbox_inches='tight')
plt.show()

## 12. Tableau récapitulatif des clusters

In [ ]:

# Profil moyen des features clés par cluster

# Permet d'interpréter chaque cluster :
#   active_day_rate     -> fréquence d'utilisation
#   max_gap_len         -> absences prolongées (vide)
#   winter_minus_summer -> saisonnalité (chauffage électrique)
#   cv_daily_kwh        -> irrégularité de la consommation
#   mean_daily_kwh      -> volume de consommation

summary_cols = ['active_day_rate', 'max_gap_len', 'winter_minus_summer',
                'cv_daily_kwh', 'mean_daily_kwh', 'r_summer']
summary = (
    features_pdl.groupby('cluster')
    .agg(n_clients=(COL_PDL, 'size'), **{c: (c, 'mean') for c in summary_cols})
    .round(3)
)
print(summary.to_string())

## 13. Comparaison avec la référence du professeur

In [ ]:

# Adjusted Rand Index vs clustering de référence

# Ce fichier n'a PAS été utilisé pour construire nos features.
# Il sert uniquement ici à mesurer l'accord entre les deux solutions.

ref = pd.read_csv(LABELS_PATH)
ref.columns = [COL_PDL, 'label_ref', 'cluster_ref']

comparison = features_pdl[[COL_PDL, 'cluster']].merge(ref, on=COL_PDL, how='inner')

ari = adjusted_rand_score(comparison['cluster_ref'], comparison['cluster'])
print(f'Adjusted Rand Index (notre clustering vs référence prof.) : {ari:.3f}')
print(f'  ARI = 1.0 -> clustering identique')
print(f'  ARI = 0.0 -> accord aléatoire')
print()

# Composition RS/RP dans chaque cluster
print('Composition RS/RP par cluster :')
comp = comparison.groupby(['cluster','label_ref']).size().unstack(fill_value=0)
comp.columns = ['RS (0)', 'RP (1)']
comp['Total'] = comp.sum(axis=1)
comp['% RP']  = (comp['RP (1)'] / comp['Total'] * 100).round(1)
print(comp.to_string())

## 14. Sauvegarde des résultats

In [ ]:

# Export pour les notebooks suivants (classification, prévision)

output = features_pdl[[COL_PDL, 'cluster']].copy()
output.to_csv('cluster_results.csv', index=False)
print(f'cluster_results.csv sauvegardé : {len(output)} clients')
print(output.head(10).to_string(index=False))